# Random Walks and PageRank, 1998 and Now

In 1998, two Stanford graduate students asked how to rank the web. Their answer, PageRank, is the long-run distribution of a bored surfer clicking random links. Thirty years later, the same algorithm with a different teleport vector is the workhorse of knowledge-graph retrieval for large language models.

This notebook builds directed graphs, runs PageRank, watches power iteration converge, and explores personalised PageRank on a toy knowledge graph of characters from the other lessons on this site.

**Accompanies the lesson:** [`/lessons/graph-theory/random-walks`](https://lutchet.netlify.app/lessons/graph-theory/random-walks)

In [1]:
import sys
sys.path.insert(0, '..')

from src.graph_theory.directed import DirectedGraph
from src.graph_theory.walks import (
    pagerank,
    personalized_pagerank,
    pagerank_steps,
    transition_matrix,
    random_walk_sample,
)
import random

## 1. A small web graph

Eight nodes, fourteen directed links. Loosely modelled on the example in Brin and Page's 1998 paper.

In [2]:
g = DirectedGraph()
edges = [
    ('A', 'B'), ('A', 'H'), ('B', 'C'), ('B', 'D'),
    ('C', 'D'), ('D', 'E'), ('D', 'C'), ('E', 'F'),
    ('E', 'D'), ('F', 'G'), ('G', 'H'), ('G', 'A'),
    ('H', 'A'), ('H', 'F'),
]
for u, v in edges:
    g.add_edge(u, v)

print('Nodes:', g.nodes)
print('Edges:', len(g.edges))

Nodes: ['A', 'B', 'H', 'C', 'D', 'E', 'F', 'G']
Edges: 14


## 2. PageRank

Power iteration with the default damping of 0.85. The ranks sum to 1, and the most-linked-to nodes bubble to the top.

In [3]:
ranks = pagerank(g)
print(f'Total mass: {sum(ranks.values()):.6f}')
print()
for node, r in sorted(ranks.items(), key=lambda kv: -kv[1]):
    bar = '#' * int(round(r * 200))
    print(f'  {node}  {r:.4f}  {bar}')

Total mass: 1.000000

  D  0.2122  ##########################################
  C  0.1384  ############################
  A  0.1191  ########################
  H  0.1191  ########################
  G  0.1171  #######################
  F  0.1157  #######################
  E  0.1089  ######################
  B  0.0694  ##############


## 3. Watch power iteration converge

`pagerank_steps` returns the rank vector after every iteration. The L1 delta shrinks rapidly for sensible damping values; convergence is usually complete in 20 to 50 iterations.

In [4]:
steps, deltas = pagerank_steps(g, damping=0.85)
print(f'Iterations to converge: {len(steps) - 1}')
print()
print('Delta per iteration:')
for i, d in enumerate(deltas[1:11], start=1):
    print(f'  iter {i:2d}: {d:.2e}')

Iterations to converge: 56

Delta per iteration:
  iter  1: 2.13e-01
  iter  2: 1.35e-01
  iter  3: 1.15e-01
  iter  4: 9.79e-02
  iter  5: 6.24e-02
  iter  6: 4.13e-02
  iter  7: 3.01e-02
  iter  8: 2.34e-02
  iter  9: 1.76e-02
  iter 10: 1.27e-02


## 4. The damping factor

Low damping flattens the ranks: teleport dominates, every node looks similar. High damping sharpens them: link structure dominates, winners win decisively. Brin and Page picked 0.85 empirically.

In [5]:
for d in [0.50, 0.70, 0.85, 0.95]:
    r = pagerank(g, damping=d)
    top = sorted(r.items(), key=lambda kv: -kv[1])[:3]
    spread = max(r.values()) - min(r.values())
    top_str = ', '.join(f'{n}={v:.3f}' for n, v in top)
    print(f'  damping {d:.2f}: top = [{top_str}]   spread = {spread:.3f}')

  damping 0.50: top = [D=0.178, C=0.130, A=0.124]   spread = 0.084
  damping 0.70: top = [D=0.197, C=0.135, A=0.122]   spread = 0.117
  damping 0.85: top = [D=0.212, C=0.138, A=0.119]   spread = 0.143
  damping 0.95: top = [D=0.223, C=0.141, A=0.116]   spread = 0.162


## 5. Sampling a random walk

If you actually let a walker loose on the graph and count where she spends her time, you recover (noisily) the PageRank vector. Below is a thirty-thousand-step walk with uniform teleport on dangling nodes. The empirical frequencies approximate the damping-1 stationary distribution.

In [6]:
rng = random.Random(42)
trajectory = random_walk_sample(g, start='A', steps=30_000, rng=rng)

from collections import Counter
visits = Counter(trajectory)
total = sum(visits.values())
print('Empirical frequencies (undamped walk):')
for node in sorted(g.nodes):
    freq = visits[node] / total
    print(f'  {node}  {freq:.4f}')

Empirical frequencies (undamped walk):
  A  0.1148
  B  0.0576
  C  0.1433
  D  0.2292
  E  0.1152
  F  0.1134
  G  0.1134
  H  0.1131


## 6. A knowledge graph

Now the other half of the story. Below is a small directed knowledge graph drawn from characters and concepts that appear in other lessons on this site. Personalised PageRank from a chosen seed returns a ranking of how related every other node is to the seed. This is the retrieval primitive in modern graph-augmented language model pipelines.

In [7]:
kg = DirectedGraph()
k_edges = [
    ('shannon', 'mtc'),
    ('mtc', 'entropy'),
    ('entropy', 'huffman_code'),
    ('huffman', 'huffman_code'),
    ('huffman_code', 'mtc'),
    ('turing', 'enigma'),
    ('bayes', 'enigma'),
    ('turing', 'complexity'),
    ('kolmogorov', 'complexity'),
    ('complexity', 'entropy'),
    ('shannon', 'entropy'),
    ('euler', 'graph_theory'),
    ('graph_theory', 'dijkstra'),
    ('graph_theory', 'pagerank'),
    ('dijkstra', 'pagerank'),
    ('entropy', 'complexity'),
    ('bayes', 'mtc'),
    ('enigma', 'shannon'),
    ('pagerank', 'graph_theory'),
    ('dijkstra', 'graph_theory'),
]
for u, v in k_edges:
    kg.add_edge(u, v)

print(f'Nodes: {len(kg.nodes)}')
print(f'Edges: {len(kg.edges)}')

Nodes: 14
Edges: 20


## 7. Personalised PageRank

Pick a seed. The walker teleports back to that node whenever she gets bored, so the resulting distribution concentrates on whatever is nearby in link structure. Different seeds light up different neighbourhoods.

In [8]:
for seed in ['shannon', 'euler', 'turing']:
    pp = personalized_pagerank(kg, seed)
    top = sorted(pp.items(), key=lambda kv: -kv[1])[:5]
    print(f'Seed: {seed}')
    for node, score in top:
        print(f'  {node:20s}  {score:.3f}')
    print()

Seed: shannon
  entropy               0.356
  mtc                   0.192
  huffman_code          0.151
  complexity            0.151
  shannon               0.150

Seed: euler
  graph_theory          0.419
  pagerank              0.254
  dijkstra              0.178
  euler                 0.150
  entropy               0.000

Seed: turing
  entropy               0.292
  complexity            0.188
  turing                0.150
  mtc                   0.128
  huffman_code          0.124



## 8. Your turn

A few things to try:

- Add a single edge to the web graph and see which node's PageRank rises the most. Can you design an edge that makes a specific target node rank first?
- Remove every outgoing edge from one node, making it dangling. How does the total mass redistribute?
- Add a new concept (say, `bernoulli`) to the knowledge graph and connect it to `bayes`. How does seeding on `bayes` change its top-5?
- Run personalised PageRank from a random seed five hundred times and average the rankings. You get (approximately) the standard PageRank back. Why?

This is the last notebook in the graph theory topic. Next up: a topic with nothing to do with graphs at all. Or so it will seem.